In [1]:
%pip install --upgrade --quiet  langchain langchain-community langchain-openai langchain-experimental neo4j wikipedia tiktoken yfiles_jupyter_graphs

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.3/114.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━

In [1]:
!pip install requests==2.32.5

In [2]:

from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [4]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')



In [5]:
from typing import Tuple, List, Optional

In [6]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser

In [7]:
from langchain_core.runnables import ConfigurableField

In [8]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [9]:
import os

In [10]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [11]:
from langchain_community.vectorstores import Neo4jVector

In [12]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [13]:
NEO4J_URI="neo4j+s://f148df90.databases.neo4j.io"
NEO4J_USERNAME="f148df90"
NEO4J_PASSWORD="avbCBIGCCgK3JSeYXhd33kSkMUPwWwFAUV73OqLjSjM"
NEO4J_DATABASE="f148df90"

In [14]:
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

In [15]:
from langchain_community.graphs import Neo4jGraph

In [16]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

with driver.session() as session:
    result = session.run("SHOW DATABASES")

    for record in result:
        print(record)

<Record name='f148df90' type='standard' aliases=[] access='read-write' address='p-mt-2a41353b6b9a-13-0034.production-orch-0695.neo4j.io:7687' role='primary' writer=True requestedStatus='online' currentStatus='online' statusMessage='' default=False home=True constituents=[]>
<Record name='system' type='system' aliases=[] access='read-write' address='p-mt-2a41353b6b9a-13-0035.production-orch-0695.neo4j.io:7687' role='primary' writer=True requestedStatus='online' currentStatus='online' statusMessage='' default=False home=False constituents=[]>
<Record name='system' type='system' aliases=[] access='read-write' address='p-mt-2a41353b6b9a-13-0034.production-orch-0695.neo4j.io:7687' role='primary' writer=False requestedStatus='online' currentStatus='online' statusMessage='' default=False home=False constituents=[]>
<Record name='system' type='system' aliases=[] access='read-write' address='p-mt-2a41353b6b9a-13-0036.production-orch-0695.neo4j.io:7687' role='primary' writer=False requestedStatu

In [17]:
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

/tmp/ipykernel_30850/2577236248.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [18]:
graph = Neo4jGraph()

In [30]:
%pip install --quiet --upgrade \
langchain \
langchain-community \
langchain-openai \
langchain-experimental \
neo4j \
wikipedia==1.4.0 \
tiktoken \
yfiles_jupyter_graphs \
requests==2.32.5

In [20]:
pip install -qU langchain_community wikipedia

In [34]:
from langchain_core.documents import Document

raw_documents = [
    Document(
        page_content="""
        Aarav is a data engineer working at TechNova in Bengaluru.
        He specializes in Apache Spark, Kafka, and Neo4j.

        Aarav collaborates with Meera, an AI engineer who builds LLM applications using LangChain and OpenAI.

        Their company recently launched a healthcare AI platform called MedGraph.
        The platform connects hospitals, doctors, and patient records using graph databases.

        Meera trained a recommendation system for Apollo Hospital.
        Aarav designed the real-time data pipeline using Kafka and BigQuery.

        TechNova partnered with Google Cloud to deploy the application.
        """,
        metadata={"source": "custom_story_1"}
    )
]

In [35]:
len(raw_documents)

1

In [36]:
raw_documents[:3]

[Document(metadata={'source': 'custom_story_1'}, page_content='\n        Aarav is a data engineer working at TechNova in Bengaluru.\n        He specializes in Apache Spark, Kafka, and Neo4j.\n        \n        Aarav collaborates with Meera, an AI engineer who builds LLM applications using LangChain and OpenAI.\n        \n        Their company recently launched a healthcare AI platform called MedGraph.\n        The platform connects hospitals, doctors, and patient records using graph databases.\n        \n        Meera trained a recommendation system for Apollo Hospital.\n        Aarav designed the real-time data pipeline using Kafka and BigQuery.\n        \n        TechNova partnered with Google Cloud to deploy the application.\n        ')]

In [40]:
pip install -U langchain-text-splitters

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=24)
documents = text_splitter.split_documents(raw_documents[:3])


In [42]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-0125")

In [43]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
llm_transformer = LLMGraphTransformer(llm=llm)

/usr/local/lib/python3.12/dist-packages/langchain_openai/chat_models/base.py:2381: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [44]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

In [45]:
graph_documents

[GraphDocument(nodes=[Node(id='Aarav', type='Person', properties={}), Node(id='Technova', type='Company', properties={}), Node(id='Bengaluru', type='Location', properties={}), Node(id='Apache Spark', type='Technology', properties={}), Node(id='Kafka', type='Technology', properties={}), Node(id='Neo4J', type='Technology', properties={}), Node(id='Meera', type='Person', properties={}), Node(id='Ai Engineer', type='Job title', properties={}), Node(id='Langchain', type='Technology', properties={}), Node(id='Openai', type='Technology', properties={}), Node(id='Medgraph', type='Platform', properties={}), Node(id='Hospital', type='Entity', properties={}), Node(id='Doctor', type='Entity', properties={}), Node(id='Patient Records', type='Entity', properties={}), Node(id='Recommendation System', type='Technology', properties={}), Node(id='Apollo Hospital', type='Hospital', properties={})], relationships=[Relationship(source=Node(id='Aarav', type='Person', properties={}), target=Node(id='Technova

In [46]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [48]:
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

In [49]:
from yfiles_jupyter_graphs import GraphWidget
from neo4j import GraphDatabase

In [50]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass

In [51]:
def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    display(widget)
    return widget

In [53]:
from typing import Tuple, List, Optional

In [54]:
from langchain_community.vectorstores import Neo4jVector

In [58]:
from langchain_openai import OpenAIEmbeddings
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)


In [59]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

[]

In [62]:
# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel,Field
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

In [63]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate

In [64]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

In [65]:
entity_chain = prompt | llm.with_structured_output(Entities)

/usr/local/lib/python3.12/dist-packages/langchain_openai/chat_models/base.py:2381: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [67]:
entity_chain.invoke({"question": "who is Aarav?"}).names

['Aarav']

In [68]:
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars

In [69]:

def generate_full_text_query(input: str) -> str:
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

In [70]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [71]:
print(structured_retriever("who is Aarav?"))

/tmp/ipykernel_30850/569629621.py:3: LangChainDeprecationWarning: The function `remove_lucene_chars` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the function exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars``.
  words = [el for el in remove_lucene_chars(input).split() if el]


Aarav - WORKS_AT -> Technova
Aarav - WORKS_AT -> Bengaluru
Aarav - SPECIALIZES_IN -> Apache Spark
Aarav - SPECIALIZES_IN -> Kafka
Aarav - SPECIALIZES_IN -> Neo4J
Aarav - DESIGNED -> Real-Time Data Pipeline


In [72]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    return final_data

In [73]:
_template = """Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question,
in its original language.
Chat History:
{chat_history}
Follow Up Input: {question}
Standalone question:"""

In [74]:
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

In [75]:
def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

In [76]:
_search_query = RunnableBranch(
    # If input includes chat_history, we condense it with the follow-up question
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),  # Condense follow-up question and chat into a standalone_question
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | ChatOpenAI(temperature=0)
        | StrOutputParser(),
    ),
    # Else, we have no chat history, so just pass through the question
    RunnableLambda(lambda x : x["question"]),
)

In [77]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""

In [78]:
prompt = ChatPromptTemplate.from_template(template)

In [79]:
chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)


In [80]:
chain.invoke({"question": "what are the technology Aarav knows?"})

Search query: what are the technology Aarav knows?


'Aarav specializes in Apache Spark, Kafka, and Neo4J.'

In [82]:
chain.invoke({"question": "does Aarav know Langchain?"})

Search query: does Aarav know Langchain?


'Yes, Aarav knows Langchain as he collaborates with Meera, who uses Langchain in her work.'

In [84]:
chain.invoke({"question": "does Aarav know Vinayaka?"})

Search query: does Aarav know Vinayaka?


'There is no information provided to determine if Aarav knows Vinayaka.'